# Les décorateurs — ajouter un comportement à une fonction sans la modifier

Tu utilises déjà des décorateurs sans le savoir : `@property`, `@staticmethod`, `@dataclass`, `@cache`...
Ce module explique **ce que fait vraiment le `@`** et comment écrire les tiens.

### L'idée en une phrase
Un **décorateur** est une fonction qui **reçoit une fonction** et **retourne une version améliorée**
de cette fonction — comme une coque de téléphone : le téléphone reste le même,
mais il est maintenant protégé (ou chronométré, ou journalisé...).

### Pourquoi c'est possible en Python
En Python, **une fonction est un objet comme un autre** : on peut la ranger dans une variable,
la passer en paramètre, la retourner. C'est la base de tout ce qui suit.

In [ ]:
# Une fonction est un objet : on peut la manipuler comme une valeur

def dire_bonjour():
    return "Bonjour !"

# 1. La ranger dans une variable (SANS parenthèses : on ne l'appelle pas, on la désigne)
salutation = dire_bonjour
print(salutation())          # Bonjour !

# 2. La passer en paramètre à une autre fonction
def executer_deux_fois(fonction):
    fonction()
    fonction()
    return "fait"

print(executer_deux_fois(print))   # passe print elle-même !

# 3. La retourner depuis une autre fonction
def fabrique_multiplicateur(n):
    def multiplier(x):
        return x * n         # multiplier « retient » la valeur de n
    return multiplier

double = fabrique_multiplicateur(2)
triple = fabrique_multiplicateur(3)
print(double(10), triple(10))      # 20 30

> Le point 3 s'appelle une **closure** (fermeture) : la fonction interne `multiplier`
> « emporte avec elle » la variable `n` de la fonction qui l'a créée, même après que celle-ci a terminé.
> C'est le mécanisme qui fait fonctionner les décorateurs.

---
## 1. Premier décorateur : un chronomètre

Structure type d'un décorateur :

```python
def mon_decorateur(fonction):          # reçoit la fonction à améliorer
    def enveloppe(*args, **kwargs):    # la « coque » qui remplacera la fonction
        # ... code AVANT l'appel ...
        resultat = fonction(*args, **kwargs)   # appel de la vraie fonction
        # ... code APRÈS l'appel ...
        return resultat
    return enveloppe                   # retourne la version améliorée
```

`*args, **kwargs` = « accepte n'importe quels arguments et transmets-les tels quels » —
ainsi la coque fonctionne avec n'importe quelle fonction.

In [1]:
import time

def chronometre(fonction):
    """Affiche le temps d'exécution de la fonction décorée."""
    def enveloppe(*args, **kwargs):
        debut = time.perf_counter()
        resultat = fonction(*args, **kwargs)
        duree = time.perf_counter() - debut
        print(f"⏱ {fonction.__name__} a pris {duree * 1000:.2f} ms")
        return resultat
    return enveloppe


@chronometre
def somme_des_carres(n):
    return sum(i ** 2 for i in range(n))


# Le @ ci-dessus est un simple raccourci pour :
#   somme_des_carres = chronometre(somme_des_carres)

total = somme_des_carres(1_000_000)
print(f"Résultat : {total}")

⏱ somme_des_carres a pris 136.31 ms
Résultat : 333332833333500000


---
## 2. `functools.wraps` — le réflexe à toujours avoir

Problème : après décoration, la fonction s'appelle... `enveloppe`. Son nom et sa documentation
(docstring) sont perdus — gênant pour le débogage et les outils.

Solution : décorer l'enveloppe avec `@functools.wraps(fonction)`, qui **recopie l'identité**
de la fonction d'origine (nom, docstring...) sur l'enveloppe. À mettre **systématiquement**.

In [ ]:
import functools

# ❌ Sans wraps : l'identité de la fonction est perdue
def sans_wraps(fonction):
    def enveloppe(*args, **kwargs):
        return fonction(*args, **kwargs)
    return enveloppe

# ✅ Avec wraps : l'identité est préservée
def avec_wraps(fonction):
    @functools.wraps(fonction)
    def enveloppe(*args, **kwargs):
        return fonction(*args, **kwargs)
    return enveloppe


@sans_wraps
def calculer_a():
    """Documentation de calculer_a."""

@avec_wraps
def calculer_b():
    """Documentation de calculer_b."""


print(f"Sans wraps : nom = {calculer_a.__name__!r}, doc = {calculer_a.__doc__!r}")
print(f"Avec wraps : nom = {calculer_b.__name__!r}, doc = {calculer_b.__doc__!r}")

---
## 3. Décorateur avec paramètres

Pour écrire `@repeter(3)`, il faut **un étage de plus** : une fonction qui reçoit le paramètre (`3`)
et qui **fabrique le décorateur**. Trois niveaux :

```
repeter(3)          → fabrique et retourne le décorateur
  └─ decorateur     → reçoit la fonction
       └─ enveloppe → le code exécuté à chaque appel
```

In [ ]:
import functools

def repeter(nombre_de_fois):
    """Fabrique un décorateur qui exécute la fonction plusieurs fois."""
    def decorateur(fonction):
        @functools.wraps(fonction)
        def enveloppe(*args, **kwargs):
            resultat = None
            for _ in range(nombre_de_fois):
                resultat = fonction(*args, **kwargs)
            return resultat
        return enveloppe
    return decorateur


@repeter(3)
def saluer(nom):
    print(f"Bonjour {nom} !")

saluer("Alice")

---
## 4. Cas concret : journaliser les appels d'une fonction

Un usage très courant en entreprise : tracer qui appelle quoi, avec quels arguments,
et ce qui en ressort — sans polluer le code métier avec des `print` partout.

In [ ]:
import functools

def journaliser(fonction):
    """Trace les appels : arguments reçus, résultat retourné, erreurs levées."""
    @functools.wraps(fonction)
    def enveloppe(*args, **kwargs):
        arguments = ", ".join(
            [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        )
        try:
            resultat = fonction(*args, **kwargs)
            print(f"[LOG] {fonction.__name__}({arguments}) → {resultat!r}")
            return resultat
        except Exception as e:
            print(f"[LOG] {fonction.__name__}({arguments}) → ⚠ {type(e).__name__}: {e}")
            raise    # on re-lève : le décorateur observe, il ne cache pas l'erreur
    return enveloppe


@journaliser
def diviser(a, b):
    return a / b

diviser(10, 2)
diviser(a=7, b=2)
try:
    diviser(1, 0)
except ZeroDivisionError:
    pass   # l'erreur a bien été journalisée puis relayée

---
## 5. Les décorateurs prêts à l'emploi qu'il faut connaître

| Décorateur | Module | Rôle |
|---|---|---|
| `@cache` / `@lru_cache(maxsize=...)` | `functools` | mémorise les résultats → évite de recalculer |
| `@cached_property` | `functools` | propriété calculée **une seule fois** puis mémorisée |
| `@property`, `@staticmethod`, `@classmethod` | intégrés | vus dans le module POO |
| `@dataclass` | `dataclasses` | vu au module précédent |
| `@wraps` | `functools` | préserve l'identité de la fonction décorée |

Démonstration de `@cache` — un cas spectaculaire pour parler **vitesse d'exécution** :

In [ ]:
import time
from functools import cache

def fibonacci_lent(n):
    if n < 2:
        return n
    return fibonacci_lent(n - 1) + fibonacci_lent(n - 2)

@cache
def fibonacci_rapide(n):
    if n < 2:
        return n
    return fibonacci_rapide(n - 1) + fibonacci_rapide(n - 2)


N = 32

debut = time.perf_counter()
fibonacci_lent(N)
t_lent = time.perf_counter() - debut

debut = time.perf_counter()
fibonacci_rapide(N)
t_rapide = time.perf_counter() - debut

print(f"Sans cache : {t_lent:.3f}s   (des millions d'appels redondants — O(2ⁿ))")
print(f"Avec cache : {t_rapide * 1000:.3f}ms (chaque valeur calculée UNE fois — O(n))")
print(f"→ le cache mémorise chaque résultat : si on redemande fibonacci(25), il ressort la réponse")
print(f"  déjà stockée au lieu de recalculer. Compromis classique : mémoire contre vitesse.")

---
## Récapitulatif

- `@decorateur` est un raccourci pour `fonction = decorateur(fonction)`.
- Un décorateur = une fonction qui reçoit une fonction et retourne une **enveloppe** améliorée.
- L'enveloppe utilise `*args, **kwargs` pour accepter n'importe quels arguments.
- **Toujours** mettre `@functools.wraps(fonction)` sur l'enveloppe.
- Décorateur paramétré (`@repeter(3)`) = un étage de plus (une fonction qui fabrique le décorateur).
- `@cache` transforme un algorithme exponentiel en linéaire — au prix de mémoire (le compromis temps/mémoire).

> **En entretien** : « Un décorateur permet d'ajouter un comportement transversal (log, cache,
> chronométrage, contrôle d'accès) sans toucher au code de la fonction — le principe
> ouvert/fermé : ouvert à l'extension, fermé à la modification. »